# 00 Next-token loss、label shift 和 perplexity

目标：不用大模型，只用小张量把 causal language model 的训练目标讲透：teacher forcing、label shift、cross entropy、`ignore_index=-100`、padding mask 和 perplexity。


## 1. 安装依赖

只需要项目根目录的基础依赖，核心是 PyTorch。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


## 2. 准备一个极小字符级数据集

真实 LLM 使用 BPE/SentencePiece tokenizer。这里用字符级 vocab，是为了让 label shift 和 loss 更直观。


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

texts = [
    "hello world",
    "hello model",
    "language model",
]

special_tokens = ["<pad>", "<bos>", "<eos>"]
chars = sorted(set("".join(texts)))
vocab = special_tokens + chars
stoi = {token: index for index, token in enumerate(vocab)}
itos = {index: token for token, index in stoi.items()}

pad_id = stoi["<pad>"]
bos_id = stoi["<bos>"]
eos_id = stoi["<eos>"]


def encode(text):
    return [bos_id] + [stoi[ch] for ch in text] + [eos_id]


def decode(ids):
    return "".join(itos[int(i)] for i in ids if int(i) >= len(special_tokens))

encoded = [encode(text) for text in texts]
max_len = max(len(row) for row in encoded)
input_ids = torch.full((len(encoded), max_len), pad_id, dtype=torch.long)
for row_index, ids in enumerate(encoded):
    input_ids[row_index, :len(ids)] = torch.tensor(ids)

print("vocab size:", len(vocab))
print("vocab:", vocab)
print("input_ids shape:", tuple(input_ids.shape))
print(input_ids)
for row in input_ids:
    print([itos[int(i)] for i in row])


## 3. label shift：输入第 t 个 token，预测第 t+1 个 token

Causal LM 训练时并不是拿 prompt 生成答案，而是对完整序列做 teacher forcing：每个位置都用真实前文预测真实下一个 token。


In [ ]:
# logits[:, t, :] 预测 labels[:, t]。
# 所以常见做法是用 input_ids[:, :-1] 作为输入，用 input_ids[:, 1:] 作为 labels。
model_inputs = input_ids[:, :-1]
labels = input_ids[:, 1:].clone()
labels[labels == pad_id] = -100

print("model_inputs shape:", tuple(model_inputs.shape))
print("labels shape:", tuple(labels.shape))

row = 0
for position in range(model_inputs.shape[1]):
    current_token = itos[int(model_inputs[row, position])]
    label_id = int(labels[row, position])
    label_token = "<ignored>" if label_id == -100 else itos[label_id]
    print(f"pos={position:02d} input={current_token!r:8s} -> label={label_token!r}")


## 4. 手算 cross entropy 和 perplexity

模型输出 logits，cross entropy 等价于对正确 token 的负 log probability 求平均。perplexity 是 `exp(loss)`，可以粗略理解成“模型平均每步有多少个困惑选择”。


In [ ]:
torch.manual_seed(0)
batch_size, seq_len = labels.shape
vocab_size = len(vocab)
logits = torch.randn(batch_size, seq_len, vocab_size)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
hf_style_loss = loss_fn(logits.reshape(-1, vocab_size), labels.reshape(-1))

log_probs = F.log_softmax(logits, dim=-1)
valid_mask = labels != -100
safe_labels = labels.clamp_min(0)
token_nll = -log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)
manual_loss = token_nll[valid_mask].mean()
perplexity = math.exp(float(manual_loss))

print("CrossEntropyLoss:", float(hf_style_loss))
print("manual loss     :", float(manual_loss))
print("perplexity      :", perplexity)
print("valid tokens    :", int(valid_mask.sum()))


## 5. padding 为什么必须 ignore

如果 padding token 也参与 loss，模型会被训练去预测一堆 `<pad>`，这会污染训练目标。SFT 中 prompt 部分置为 `-100` 也是同一个机制。


In [ ]:
labels_without_ignore = input_ids[:, 1:].clone()
loss_with_pad = loss_fn(logits.reshape(-1, vocab_size), labels_without_ignore.reshape(-1))
loss_without_pad = loss_fn(logits.reshape(-1, vocab_size), labels.reshape(-1))

print("loss including pad tokens:", float(loss_with_pad))
print("loss ignoring pad tokens :", float(loss_without_pad))
print("pad labels count:", int((labels_without_ignore == pad_id).sum()))


## 6. 训练一个极小 next-token 模型

这个模型很小，不追求效果，只看 loss 是否按 next-token objective 下降。


In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, hidden_size=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids):
        hidden = self.embed(input_ids)
        logits = self.lm_head(hidden)
        return logits


torch.manual_seed(1)
model = TinyCausalLM(len(vocab))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

for step in range(101):
    optimizer.zero_grad()
    logits = model(model_inputs)
    loss = loss_fn(logits.reshape(-1, len(vocab)), labels.reshape(-1))
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(f"step={step:03d} loss={float(loss):.4f} ppl={math.exp(float(loss)):.2f}")


## 7. 用训练好的 toy model 生成

训练时模型每一步都看到真实前文；推理时它只能看到自己已经生成的 token。这就是训练和推理之间常见的 teacher forcing 差异。


In [ ]:
def generate(prefix, max_new_tokens=20):
    ids = [bos_id] + [stoi[ch] for ch in prefix]
    for _ in range(max_new_tokens):
        x = torch.tensor(ids, dtype=torch.long).unsqueeze(0)
        with torch.inference_mode():
            next_logits = model(x)[:, -1, :]
        next_id = int(next_logits.argmax(dim=-1))
        ids.append(next_id)
        if next_id == eos_id:
            break
    return ids, decode(ids)

for prefix in ["h", "hello ", "language "]:
    ids, text = generate(prefix)
    print(prefix, "->", [itos[i] for i in ids], "->", text)


## 面试总结

- Causal LM 的核心训练目标是 next-token prediction。
- label shift：`input_ids[:, :-1]` 输入，`input_ids[:, 1:]` 作为 labels。
- teacher forcing：训练时每个位置都使用真实历史 token。
- cross entropy 是正确 token 的负 log probability 平均值。
- perplexity = `exp(loss)`，loss 越低 perplexity 越低。
- padding、prompt、不想监督的 token 都可以用 `ignore_index=-100` 排除出 loss。
- 推理时模型吃的是自己前一步生成的 token，所以训练 loss 低不等于长文本生成一定稳定。
